# Redes Neuronales Convolucionales (CNN) - ML Journey

## ¿Qué son las Redes Convolucionales?

Las Redes Neuronales Convolucionales (CNN) son un tipo especializado de red neuronal diseñada específicamente para procesar datos que tienen una estructura de cuadrícula, como las **imágenes**.

### ¿Por qué CNNs para imágenes?

Las redes neuronales tradicionales (como las del notebook anterior) tienen algunas limitaciones con imágenes:

1. **Pérdida de estructura espacial**: Al "aplanar" una imagen, perdemos la información sobre qué píxeles están cerca de cuáles
2. **Muchos parámetros**: Una imagen de 28x28 tiene 784 píxeles, crear conexiones densas requiere millones de parámetros
3. **No son invariantes a traslación**: Si movemos un objeto en la imagen, la red podría no reconocerlo

### Conceptos Fundamentales de CNNs:

#### 1. **Convolución**
- Operación que aplica un **filtro** (kernel) sobre la imagen
- Detecta características como bordes, líneas, texturas
- Preserva la estructura espacial de la imagen

#### 2. **Filtros/Kernels**
- Pequeñas matrices (ej. 3x3) que se "deslizan" sobre la imagen
- Cada filtro aprende a detectar una característica específica
- Ejemplos: detector de bordes horizontales, verticales, diagonales

#### 3. **Pooling (Submuestreo)**
- Reduce el tamaño de la imagen manteniendo la información importante
- **Max Pooling**: Toma el valor máximo en cada región
- **Average Pooling**: Toma el promedio en cada región

#### 4. **Arquitectura Típica**
```
INPUT → CONV → POOLING → CONV → POOLING → ... → FLATTEN → DENSE → OUTPUT
```

### Ventajas de las CNNs:
- ✅ **Invariantes a traslación**: Reconocen objetos sin importar su posición
- ✅ **Menos parámetros**: Comparten pesos entre diferentes posiciones
- ✅ **Jerarquía de características**: Detectan desde líneas hasta objetos completos
- ✅ **Preservan estructura espacial**: Mantienen las relaciones entre píxeles cercanos

In [ ]:
import numpy as np
from scipy import datasets
import matplotlib.pyplot as plt

# Cargamos una imagen de ejemplo (escalón ascendente)
img = datasets.ascent().astype(float)

print("=== IMAGEN ORIGINAL ===")
print(f"Dimensiones de la imagen: {img.shape}")
print(f"Rango de valores: {img.min()} - {img.max()}")

# Visualizamos la imagen original
plt.figure(figsize=(10, 8))
plt.imshow(img, cmap='gray')
plt.title('Imagen Original')
plt.axis('off')
plt.colorbar()
plt.show()

# APLICANDO FILTROS MANUALMENTE

In [ ]:
print("=== APLICANDO FILTROS DE CONVOLUCIÓN ===")

# Creamos una copia de la imagen para transformar
img_transformed = np.copy(img)
size_x = img_transformed.shape[0]
size_y = img_transformed.shape[1]

# Definimos diferentes tipos de filtros

# Filtro 1: Blur/Suavizado (reduce ruido)
filter = [[1, 2, 1],
               [2, 4, 2],
               [1, 2, 1]]

# Filtro 2: Detector de bordes horizontales
# filter = [[-1, -2, -1],
#           [ 0,  0,  0],
#           [ 1,  2,  1]]

# Filtro 3: Detector de bordes verticales
# filter = [[-1, 0, 1],
#           [-2, 0, 2],
#           [-1, 0, 1]]

In [ ]:
# Usamos el filtro de blur para esta demostración
weight = 1 / 8  # Peso de normalización para el filtro blur

print("Aplicando convolución manualmente...")
print(f"Filtro utilizado: {filter}")
print(f"Peso de normalización: {weight}")

# Aplicamos la convolución manualmente píxel por píxel
for x in range(1, size_x - 1):  # Empezamos en 1 y terminamos en size-1 para evitar bordes
    for y in range(1, size_y - 1):
        # Calculamos la convolución para el píxel (x, y)
        convolution = 0.0

        # Multiplicamos cada elemento del filtro con el píxel correspondiente
        # y sumamos todos los resultados
        convolution = convolution + (img[x - 1, y - 1] * filter[0][0])  # Esquina superior izquierda
        convolution = convolution + (img[x, y - 1] * filter[0][1])      # Arriba
        convolution = convolution + (img[x + 1, y - 1] * filter[0][2])  # Esquina superior derecha
        convolution = convolution + (img[x - 1, y] * filter[1][0])      # Izquierda
        convolution = convolution + (img[x, y] * filter[1][1])          # Centro
        convolution = convolution + (img[x + 1, y] * filter[1][2])      # Derecha
        convolution = convolution + (img[x - 1, y + 1] * filter[2][0])  # Esquina inferior izquierda
        convolution = convolution + (img[x, y + 1] * filter[2][1])      # Abajo
        convolution = convolution + (img[x + 1, y + 1] * filter[2][2])  # Esquina inferior derecha

        # Aplicamos el peso de normalización
        convolution = convolution * weight

        # Aplicamos función de activación ReLU (valores negativos = 0)
        if convolution < 0:
            convolution = 0
        # Limitamos valores muy altos
        if convolution > 255:
            convolution = 255

        # Asignamos el nuevo valor al píxel
        img_transformed[x, y] = convolution

# Visualizamos el resultado
plt.figure(figsize=(10, 8))
plt.imshow(img_transformed, cmap='gray')
plt.title('Imagen después de Aplicar Convolución (Filtro Blur)')
plt.axis('off')
plt.colorbar()
plt.show()

In [ ]:
print("=== APLICANDO MAX POOLING ===")

import skimage.measure

# Mostramos la imagen antes del pooling
plt.figure(figsize=(25, 10))

plt.subplot(1, 3, 1)
plt.imshow(img_transformed, cmap='gray')
plt.title('Antes del Pooling')
plt.axis('off')

print(f"Tamaño antes del pooling: {img_transformed.shape}")

# Aplicamos Max Pooling con ventanas de 2x2
img_pooled = skimage.measure.block_reduce(img_transformed, (2, 2), np.max)

plt.subplot(1, 3, 2)
plt.imshow(img_pooled, cmap='gray')
plt.title('Después de Max Pooling (2x2)')
plt.axis('off')

print(f"Tamaño después del pooling: {img_pooled.shape}")

In [ ]:
# Aplicamos otro Max Pooling para mostrar el efecto acumulativo
img_pooled_2 = skimage.measure.block_reduce(img_pooled, (2, 2), np.max)

plt.imshow(img_pooled_2, cmap='gray')
plt.title('Después de 2 Max Poolings')
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
print(f"Tamaño después del segundo pooling: {img_pooled_2.shape}")
print("📉 Observa cómo se reduce el tamaño pero se mantiene la información esencial")

# CONSTRUCCIÓN Y ENTRENAMIENTO DE UNA CNN

In [ ]:
import tensorflow as tf

print("\n" + "="*60)
print("CNN COMPLETA CON FASHION MNIST")
print("="*60)

# Cargamos Fashion MNIST (mismos datos del notebook anterior)
fashion_mnist = tf.keras.datasets.fashion_mnist
(training_images, training_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Normalizamos las imágenes
training_images = training_images / 255.0
test_images = test_images / 255.0

# Nombres de las clases
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Datos cargados: {training_images.shape[0]} imágenes de entrenamiento")
print(f"Cada imagen es de: {training_images.shape[1]}x{training_images.shape[2]} píxeles")

In [ ]:
# Creamos el modelo CNN
cnn_model = tf.keras.models.Sequential([
    # Capa de entrada: especificamos las dimensiones de la imagen
    tf.keras.layers.Input((28, 28, 1)),  # 28x28 píxeles, 1 canal (escala de grises)

    # BLOQUE CONVOLUCIONAL 1
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),  # 64 filtros de 3x3
    tf.keras.layers.MaxPooling2D(2, 2),                     # Max pooling de 2x2

    # BLOQUE CONVOLUCIONAL 2
    tf.keras.layers.Conv2D(256, (3, 3), activation='relu'), # 256 filtros de 3x3
    tf.keras.layers.MaxPooling2D(2, 2),                     # Max pooling de 2x2

    # TRANSICIÓN A RED DENSA
    tf.keras.layers.Flatten(),          # Aplanamos para conectar con capas densas

    # CAPAS DENSAS (como en el notebook anterior)
    tf.keras.layers.Dense(128, activation='relu'),  # Capa oculta densa
    tf.keras.layers.Dense(10, activation='softmax') # Capa de salida (10 clases)
])

In [ ]:
print("Arquitectura de la CNN:")
cnn_model.summary()

# Compilamos el modelo
cnn_model.compile(
    optimizer=tf.optimizers.SGD(),              # Mismo optimizador que antes
    loss='sparse_categorical_crossentropy',     # Misma función de pérdida
    metrics=['accuracy']                        # Misma métrica
)

print("\n✅ Modelo CNN compilado exitosamente")

In [ ]:
print("\n=== ENTRENANDO LA CNN ===")
print("⏰ Esto tomará más tiempo que la red neuronal simple...")

# Entrenamos la CNN (menos épocas para este demo)
cnn_history = cnn_model.fit(
    training_images.reshape(-1, 28, 28, 1),  # Reshape para agregar dimensión de canal
    training_labels,
    epochs=3,        # Solo 3 épocas para el demo (en la vida real usarías más)
    verbose=1
)

In [ ]:
print("\n=== EVALUACIÓN DE LA CNN ===")

# Evaluamos la CNN
test_images_reshaped = test_images.reshape(-1, 28, 28, 1)
cnn_loss, cnn_accuracy = cnn_model.evaluate(test_images_reshaped, test_labels, verbose=0)

print(f"CNN - Precisión en conjunto de prueba: {cnn_accuracy:.4f} ({cnn_accuracy*100:.2f}%)")

# Para comparar, recordemos los resultados de la red neuronal simple del notebook anterior
print(f"\n📊 COMPARACIÓN APROXIMADA:")
print(f"Red Neuronal Simple (del notebook anterior): ~83-90%")
print(f"CNN (este modelo): {cnn_accuracy*100:.2f}%")

if cnn_accuracy > 0.88:
    print("🎉 ¡La CNN probablemente obtuvo mejores resultados!")
else:
    print("🤔 Los resultados varían - prueba con más épocas de entrenamiento")

In [ ]:
print("\n" + "="*60)
print("RESUMEN DEL MÓDULO CNNs")
print("="*60)

print("\n🧠 CONCEPTOS APRENDIDOS:")
print("1. ¿Qué son las convoluciones y cómo funcionan?")
print("2. Diferentes tipos de filtros (blur, detección de bordes)")
print("3. ¿Qué es el pooling y para qué sirve?")
print("4. Arquitectura completa de una CNN")
print("5. Comparación entre CNNs y redes neuronales tradicionales")

print("\n🛠️ HABILIDADES TÉCNICAS:")
print("1. Aplicar convoluciones manualmente")
print("2. Implementar max pooling")
print("3. Construir CNNs con TensorFlow/Keras")
print("4. Entrenar y evaluar CNNs")
print("5. Comparar diferentes arquitecturas")

print("\n📈 VENTAJAS DE LAS CNNs QUE HEMOS VISTO:")
print("1. ✅ Mejor rendimiento en imágenes")
print("2. ✅ Menos parámetros que redes densas equivalentes")
print("3. ✅ Invariancia a traslación")
print("4. ✅ Detección jerárquica de características")
print("5. ✅ Preservación de estructura espacial")

print("\n🚀 PRÓXIMOS PASOS SUGERIDOS:")
print("1. Experimentar con diferentes arquitecturas")
print("2. Probar técnicas de aumento de datos")
print("3. Explorar CNNs más avanzadas (ResNet, VGG, etc.)")
print("4. Aplicar CNNs a otros datasets (CIFAR-10, ImageNet)")
print("5. Aprender sobre transfer learning")

print("\n🎯 ¡FELICITACIONES!")
print("Has completado una introducción sólida a las redes neuronales convolucionales.")
print("Ahora tienes las bases para explorar arquitecturas más avanzadas de deep learning.")